In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())

print("Click here to download the motion-dataset.zip: ", FileLink("motion-dataset.zip"))


Click here to download the motion-dataset.zip:  /notebooks/motion-synthesis/motion-dataset.zip


In [2]:
import torch
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE
from networks.trainers import MotionVQVAETrainer

In [3]:
parser = TrainOptions()
options = parser.parse(args = [])
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 196
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

In [14]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_file = pjoin(options.data_root, 'train_micro.txt')
val_split_file = pjoin(options.data_root, 'val_micro.txt')

train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)
st = set()
for ds in train_dataset:
    st.add(ds['motion_parts'].shape)
print(st)
sample_motion = train_dataset[105]
print('Sample data shape: ', sample_motion['motion_parts'].shape)
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 8


100%|██████████| 8/8 [00:00<00:00, 7376.22it/s]


Motion shape (B, T, D): (8, 199, 263)
Total number of motions 8, snippets 468
id list 4


100%|██████████| 4/4 [00:00<00:00, 4953.41it/s]

Motion shape (B, T, D): (4, 170, 263)
Total number of motions 4, snippets 495
{(40, 6, 60)}
Sample data shape:  (40, 6, 60)


In [5]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=True, num_workers=1,
                              shuffle=False, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=True, num_workers=1,
                        shuffle=False, pin_memory=True)

vqvae = MotionVQVAE(
    input_dim=Dp_max,
    enc_hidden_dim=256,
    dec_hidden_dim=256,
    latent_dim=256,
    num_embeddings=256,
    beta=0.25
)

trainer = MotionVQVAETrainer(options, vqvae = vqvae)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader)


Iters Per Epoch, Training: 0003, Validation: 003
Validation time:
Validation Loss: 0.46389 Reconstruction Loss: 0.46389 VQ Loss: 0.00000 Codebook Loss: 0.00000 Commitment Loss: 0.00000
epoch: 001 inner_iter:     1 0m 13s (- 1m 7s) niter: 0000005 completed:  16%) val_loss: 0.4639  loss: 0.8342  loss_rec: 0.8342  loss_vq: 0.0000  loss_codebook: 0.0000  loss_commit: 0.0000 
Validation time:
Validation Loss: 0.46182 Reconstruction Loss: 0.46182 VQ Loss: 0.00000 Codebook Loss: 0.00000 Commitment Loss: 0.00000
Validation time:
Validation Loss: 0.45906 Reconstruction Loss: 0.45906 VQ Loss: 0.00000 Codebook Loss: 0.00000 Commitment Loss: 0.00000
epoch: 003 inner_iter:     0 0m 38s (- 1m 16s) niter: 0000010 completed:  33%) val_loss: 0.4591  loss: 0.8500  loss_rec: 0.8500  loss_vq: 0.0000  loss_codebook: 0.0000  loss_commit: 0.0000 
Validation time:
Validation Loss: 0.45622 Reconstruction Loss: 0.45622 VQ Loss: 0.00000 Codebook Loss: 0.00000 Commitment Loss: 0.00000
epoch: 004 inner_iter:     2